In [1]:
!apt-get update
!apt-get install zstd -y
!curl -fsSL https://ollama.com/install.sh | sh
# ollama serve ## This command in terminal

Get:1 http://archive.ubuntu.com/ubuntu jammy InRelease [270 kB]
Get:2 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:3 http://security.ubuntu.com/ubuntu jammy-security/multiverse amd64 Packages [62.6 kB]
Get:4 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:5 http://security.ubuntu.com/ubuntu jammy-security/universe amd64 Packages [1311 kB]
Get:6 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Get:7 http://archive.ubuntu.com/ubuntu jammy/universe amd64 Packages [17.5 MB]
Get:8 http://archive.ubuntu.com/ubuntu jammy/main amd64 Packages [1792 kB]     
Get:9 http://security.ubuntu.com/ubuntu jammy-security/restricted amd64 Packages [7032 kB]
Get:10 http://archive.ubuntu.com/ubuntu jammy/multiverse amd64 Packages [266 kB]
Get:11 http://security.ubuntu.com/ubuntu jammy-security/main amd64 Packages [3937 kB]
Get:12 http://archive.ubuntu.com/ubuntu jammy/restricted amd64 Packages [164 kB]
Get:13 http://archive.ubuntu.com/ubunt

In [2]:
!ollama pull mistral
!ollama pull llama3.1:8b

]11;?\pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest ⠼ pulling manifest ⠴ pulling manifest ⠦ pulling manifest ⠧ pulling manifest ⠇ pulling manifest ⠏ pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest ⠼ pulling manifest ⠴ pulling manifest ⠦ pulling manifest ⠧ pulling manifest ⠇ pulling manifest ⠏ pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest ⠼ pulling manifest ⠴ pulling manifest ⠦ pulling manifest ⠧ pulling manifest ⠇ pulling manifest ⠏ pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest ⠼ pulling manifest ⠴ pulling manifest ⠦ pulling manifest ⠧ pulling manifest 
pulling f5074b1221da:   0% ▕                  ▏ 2.1 MB/4.4 GB                  pulling manifest 
pulling f5074b1221da:   0% ▕                  ▏ 6.6 MB/4.4 GB                  pulling manifest 
pulling f5074b1221da:   0% ▕                  ▏  15 MB/4.4

In [3]:
!pip install langchain==0.3.28 langchain-community langchain-core chromadb numpy==1.26.4 langchain-huggingface sentence-transformers langchain-chroma langchain-openai pypdf langgraph ddgs

Looking in indexes: https://pypi.org/simple, https://pypi.ngc.nvidia.com
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 88.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 108.8/108.8 kB 127.0 MB/s eta 0:00:00
INFO: pip is looking at multiple versions of langchain-community to determine which version is compatible with other requirements. This could take a while.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 209.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.0/42.0 kB 175.7 MB/s eta 0:00:00
INFO: pip is looking at multiple versions of langchain-huggingface to determine which version is compatible with other requirements. This could take a while.
INFO: pip is looking at multiple versions of langchain-chroma to determine which version is compatible with other requirements. This could take a while.
INFO: pip is looking at multiple versions of langchain-openai to determine which version is compatible with other requirements. This cou

In [1]:
# Standard library imports
import os
import subprocess
import urllib.parse
import urllib.request
import xml.etree.ElementTree as ET
from urllib.parse import quote, urlparse

# Third-party imports
import torch
import requests
import chromadb
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline

# LangChain imports
from langchain.tools import tool
from langchain.schema import Document
from langchain.prompts import PromptTemplate
from langchain.agents import initialize_agent, Tool
from langchain_community.document_loaders import PyPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.agents import AgentExecutor, create_react_agent
from langchain_core.prompts import PromptTemplate

# LangChain Community integrations
from langchain_community.llms import Ollama
from langchain_community.vectorstores import Chroma
from langchain_community.embeddings import OllamaEmbeddings
from langchain_community.chat_models import ChatOllama

# Chroma specific

from chromadb import PersistentClient

import warnings
warnings.filterwarnings("ignore", category=UserWarning, module="pdfminer")

/usr/local/lib/python3.10/dist-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Disabling PyTorch because PyTorch >= 2.4 is required but found 2.2.0a0+81ea7a4
PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.
PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


In [2]:
# llm_model="mistral"
embedding_model = "mistral"
llama_model="llama3.1:8b"

# llm = Ollama(model=llm_model)
summarizer = Ollama(model=llama_model)
llama = Ollama(model=llama_model)

/tmp/ipykernel_928/77737685.py:6: LangChainDeprecationWarning: The class `Ollama` was deprecated in LangChain 0.3.1 and will be removed in 1.0.0. An updated version of the class exists in the :class:`~langchain-ollama package and should be used instead. To use it run `pip install -U :class:`~langchain-ollama` and import as `from :class:`~langchain_ollama import OllamaLLM``.
  summarizer = Ollama(model=llama_model)


In [3]:
persist_dir = "chroma_store"
embedding_fn = OllamaEmbeddings(model=embedding_model)
chroma_client = chromadb.PersistentClient(path=persist_dir)
vdb_available = False

available_collections = chroma_client.list_collections()

device = "cuda" if torch.cuda.is_available() else "cpu" 

/tmp/ipykernel_928/3017214188.py:2: LangChainDeprecationWarning: The class `OllamaEmbeddings` was deprecated in LangChain 0.3.1 and will be removed in 1.0.0. An updated version of the class exists in the :class:`~langchain-ollama package and should be used instead. To use it run `pip install -U :class:`~langchain-ollama` and import as `from :class:`~langchain_ollama import OllamaEmbeddings``.
  embedding_fn = OllamaEmbeddings(model=embedding_model)


In [4]:
def extract_name(url):
    parsed = urlparse(url)
    filename = os.path.basename(parsed.path)
    return filename

In [5]:
def download_pdf(pdf_link: str, filename: str, save_dir="pdf"):
    # Ensure filename ends with .pdf
    if not filename.lower().endswith(".pdf"):
        filename += ".pdf"
    
    # Ensure directory exists
    os.makedirs(save_dir, exist_ok=True)
    
    save_path = os.path.join(save_dir, filename)

    # Stream download
    response = requests.get(pdf_link, stream=True)
    response.raise_for_status()
    
    with open(save_path, "wb") as f:
        for chunk in response.iter_content(chunk_size=8192):
            f.write(chunk)
    
    return f"{save_dir}/{filename}"

def load_and_split_pdf(pdf_path, author_name, published_on, summary, chunk_size=500, chunk_overlap=50):
    loader = PyPDFLoader(pdf_path)
    documents = loader.load()

    for d in documents:
        d.metadata["author"] = author_name
        d.metadata["published_on"] = published_on
        d.metadata["abstract"] = summary
        
    
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap
    )
    return splitter.split_documents(documents)

def create_and_store_embeddings(docs, pdf_name):
    # Create vectorstore (new collection for each PDF)
    vectorstore = Chroma.from_documents(
        docs,
        embedding=embedding_fn,
        collection_name=pdf_name,
        persist_directory=persist_dir
    )

    print(f"\nStored {len(docs)} chunks in collection '{pdf_name}'")

In [6]:
@tool("arxiv")
def arxiv(search_query: str,
        from_date: str = "20000101", 
        to_date: str = "20250601", 
        max_results: int = 3, 
        PDF_FOLDER: str = "pdf"
         ):
    """
    Search arXiv for papers, download PDFs, process them, and store embeddings.
    """
    global vdb_available
    vdb_available = False 
    
    try:
        chroma_client = chromadb.PersistentClient(path=persist_dir)
        pre = chroma_client.count_collections()
        available_collections = chroma_client.list_collections()
        collection_names = [col.name for col in available_collections]
        
        new_collection = []
        search = f'all:"{search_query}" AND submittedDate:[{from_date} TO {to_date}]'
        encoded_query = urllib.parse.quote(search)

        url = f"http://export.arxiv.org/api/query?search_query={encoded_query}&max_results={max_results}&sortBy=submittedDate&sortOrder=descending"
        with urllib.request.urlopen(url) as response:
            data = response.read()

        root = ET.fromstring(data)
        ns = {"atom": "http://www.w3.org/2005/Atom"}
        entries = root.findall("atom:entry", ns)

        if not entries:
            return (
                f"\nNo results were found on arXiv for your query: '{search_query}'.\n"
                "This tool searches research papers using the arXiv API. "
                "Try rephrasing your query with broader or different keywords, "
                "or switch to the general Search tool for more sources.\n"
            )

        for entry in entries:
            title = entry.find("atom:title", ns).text.strip()
            print("Got Paper Entitled: ", title)
            summary = entry.find("atom:summary", ns).text.strip()
            published = entry.find("atom:published", ns).text.strip()
    
            authors = [author.find("atom:name", ns).text.strip()
                       for author in entry.findall("atom:author", ns)]
    
            pdf_link = None
            for link in entry.findall("atom:link", ns):
                if link.attrib.get("title") == "pdf":
                    pdf_link = link.attrib["href"]
                    break
    
            if pdf_link:
                filename = extract_name(pdf_link)
                
                if filename in collection_names:
                    print(f"\nPDF {filename} already exists.")
                else:
                    pdf_path = download_pdf(pdf_link, filename, PDF_FOLDER)
                    docs = load_and_split_pdf(
                        pdf_path=pdf_path,
                        author_name=", ".join(authors),
                        published_on=published,
                        summary=summary
                    )
                    create_and_store_embeddings(docs=docs, pdf_name=filename)
                    new_collection.append(filename)

                vdb_available = True
            else:
                print(f"\nNo PDF link found for: {title}")

        if pre < chroma_client.count_collections():
            return (
                f"\nLocal collections updated with:\n{new_collection}\n\n"
                f"Available collections:\n{chroma_client.list_collections()}\n\n"
            )
        elif len(new_collection) == 0:
            return "\nNo new collections were added (all already existed).\n"
        else:
            return (
                f"\n{new_collection} already existed in the database.\n\n"
                f"Available collections:\n{chroma_client.list_collections()}\n\n"
            )

    except Exception as e:
        return f"\nError searching arXiv: {str(e)}\n"


In [7]:
def search_across_all(query, n_results=5):
    client = chromadb.PersistentClient(path="chroma_store/")
    all_collections = client.list_collections()

    query_vector = embedding_fn.embed_query(query)
    search_results = {}

    for col in all_collections:
        results = client.get_collection(col.name).query(
            query_embeddings=[query_vector],
            n_results=n_results
        )
        if results["documents"]:
            search_results[col.name] = results

    return search_results

In [8]:
@tool("question_answring_from_vdb")
def question_answring_from_vdb(
    question: str,
    per_collection_k: int = 10,
    max_new_tokens: int = 500
):
    """
    Answer user questions based on the documents stored in the vector database.

    This tool performs a similarity search across all collections in the vector DB,
    retrieves the most relevant document chunks, and uses a Gemini model
    to generate a detailed answer. If the answer is not found in the database,
    it will fallback to ArxivSearchTool.
    """

    # 1. Run search across all collections
    results = search_across_all(question, n_results=per_collection_k)

    # 2. Handle "no DB" or "no relevant results" messages
    if isinstance(results, dict) and "message" in results:
        return f"\n{results['message']} Falling back to Arxiv Tool...\n"

    # 3. Extract only the document texts
    context_docs = []
    for col_name, res in results.items():
        docs = res.get("documents", [[]])[0]
        for d in docs:
            context_docs.append(f"[{col_name}] {d}")

    # 4. If nothing found, fallback to ArxivSearchTool
    if not context_docs:
        return f"\nNo relevant context found in collections. Try to research tool\n"

    # 5. Build context string
    context = "\n\n".join(context_docs)

    # 6. Build the prompt
    prompt = f"""
    You are an expert assistant.
    Use the following context to answer the question in detail.
    If the answer is not in the context, say the provided context don't have such information.

    Context:
    {context}

    Question: {question}

    Answer:
    """
    
    # 8. Generate answer
    answer = llama.invoke(prompt)
    return f"\n{answer.strip()}\n"

In [9]:
@tool("summarize_from_keyword")
def summarize_from_keyword(keyword: str, top_k: int = 5):
    """
    Summarize content related to a keyword by retrieving chunks from all Chroma collections
    (via search_across_all) and summarizing them with Gemini.
    """

    # Step 1: Retrieve from all collections
    results = search_across_all(keyword, n_results=top_k)

    # Step 2: Extract documents
    retrieved_chunks = []
    for col_name, res in results.items():
        docs = res.get("documents", [[]])[0]  # take first sublist (1 query embedding)
        for d in docs:
            retrieved_chunks.append(f"[{col_name}] {d}")

    # Step 3: Fallback if nothing found
    if not retrieved_chunks:
        retrieved_chunks = ["No relevant context found."]

    # Step 4: Build context
    context_text = " ".join(retrieved_chunks)

    # Step 5: Summarization prompt
    prompt = f"""
    You are a helpful assistant. Summarize the following text in relation to the keyword '{keyword}'.
    Focus only on relevant information and keep it concise.

    Context:
    {context_text}
    """

    # Step 7: Call Gemini
    # response = gemini.invoke(prompt)
    response = llama.invoke(prompt)

    return f"\n{getattr(response, 'content', str(response))}\n"


In [10]:
def get_citations_from_results(results: dict):
    """
    Convert search_across_all() results into citation-style dictionary.

    Args:
        results (dict): Output of search_across_all(query)

    Returns:
        dict: citation dictionary keyed by paper id
    """
    citations = {}

    for source_id, res in results.items():
        # res["metadatas"] is a list of lists -> take first list
        metadatas = res.get("metadatas", [[]])[0]

        if not metadatas:
            continue

        # use first metadata (each chunk has same paper-level info)
        meta = metadatas[0]

        citations[source_id] = {
            "id": source_id,
            "title": meta.get("title", "Unknown Title"),
            "authors": meta.get("author", meta.get("author_name", "Unknown Author")),
            "published_date": meta.get("published_on", "Unknown Date"),
            "doi": meta.get("doi", ""),
            "arxiv": meta.get("arxivid", ""),
        }

    return citations

In [11]:
def format_citations_apa(citations: dict):
    """
    Format citations in APA style.
    Expects dict from get_citations_from_results()
    """
    formatted = []
    i=1
    for meta in citations.values():
        authors = meta.get("authors", "Unknown Author")
        year = meta.get("published_date", "n.d.")[:4]  # extract year
        title = meta.get("title", "Untitled")
        doi = meta.get("doi", "")
        arxiv = meta.get("arxiv", "")

        entry = f"[{i}] {authors} ({year}). {title}. {doi if doi else arxiv}"
        i+=1
        formatted.append(entry)

    return "\n".join(formatted)

In [12]:
@tool("get_citations")
def get_citations(keyword):
    """
    Retrieve and format citations from the vector database.
    """
    try:
        results = search_across_all(keyword, n_results=3)
        if isinstance(results, dict) and "message" in results:
            return results["message"]
        
        citations = get_citations_from_results(results)
        formatted = format_citations_apa(citations)
#         print(format_citations_apa(citations))
        return f"\n{formatted}\n"
    except Exception as e:
        return f"\nError retrieving citations: {str(e)}\n"

In [13]:
@tool("get_insights")
def get_insights(query: str, top_k: int = 5, max_new_tokens: int = 400):
    """
    Retrieve insights from the database based on a query using Gemini.

    Args:
        query (str): User query to search across all collections.
        top_k (int): Number of top chunks to retrieve per collection.
        max_new_tokens (int): Max tokens for the response.

    Returns:
        str: Insightful answer generated by Gemini.
    """

    # 1. Retrieve relevant chunks
    results = search_across_all(query, n_results=top_k)

    # 2. Extract document text
    retrieved_chunks = []
    for col_name, res in results.items():
        docs = res.get("documents", [[]])[0]
        for d in docs:
            retrieved_chunks.append(f"[{col_name}] {d}")

    # 3. If no chunks found, return fallback
    if not retrieved_chunks:
        return "\nNo relevant information found in the database.\n"

    # 4. Build context
    context_text = "\n\n".join(retrieved_chunks)

    # 5. Build prompt
    prompt = f"""
    You are an expert analyst. I will provide you with a context.
Your task is to generate insightful analysis only if the context provides enough information.
If there is no meaningful analysis possible, simply respond with "No significant insights found."

When analysis is possible, structure your response under the following headings:

Applications & Use Cases
- Where can this be applied effectively?
- What real-world scenarios or industries can benefit?

Advantages & Differentiation
- Why is this better compared to alternatives?
- What makes it unique, impactful, or valuable?

Limitations & Challenges
- What risks, drawbacks, or constraints should be considered?

Trends & Patterns
- Does the context reveal any emerging trends, shifts, or long-term implications?

Actionable Takeaways
- What practical recommendations, strategies, or next steps can be derived?

Guidelines for response:
- Be structured and concise.
- Provide high-value insights, not surface-level summaries.
- Skip any section where no meaningful analysis is possible.

Context:
{context_text}

Query: {query}

Provide clear, structured insights (not just a summary).
    """

    # 6. Initialize Gemini client locally (avoids deepcopy/pickle errors)
    # gemini = ChatGoogleGenerativeAI(
    #     model="gemini-2.5-flash",
    #     temperature=0.3,
    #     google_api_key=API_KEY
    # )

    # 7. Call Gemini
    # response = gemini.invoke(prompt)
    response = llama.invoke(prompt)

    return f"\n{getattr(response, 'content', str(response))}\n"

In [14]:
vdb_router_prompt = PromptTemplate(
    template="""
You are a router agent for a set of tools that retrieve data from the vector database to answer user queries.

Answer the as best you can using the available tools.
Provide a detailed and comprehensive final answer. 
Include background, problem statement, methodology, architecture details, experimental setup, results, comparisons, 
impact on future research, and industry applications with citations of related paper in the last. 


You must follow EXACTLY one of these two options in your output:

Option 1 (if you need to use a tool):
Question: the input question you must answer
Thought: reasoning about which tool to use

When you decide to use a tool, you must output ONLY the following lines (nothing else):

Action: <tool name, exactly as given in [{tool_names}]>
Action Input: <input for the tool>

Do NOT add any extra text, explanation, or commentary after Action or Action Input.

(then wait for the Observation. You may repeat this Thought/Action/Action Input/Observation cycle N times,
but never repeat the same tool once it has been used.)

Option 2 (if you are ready to give the final response):
Thought: I now know the final answer

When you give the final answer, you must output ONLY:

Final Answer: <your detailed response here>

Rules:
- NEVER output both an Action and a Final Answer in the same step.
- Use clear, complete explanations in the Final Answer (not minimal words).

Begin!

Question: {input}
{agent_scratchpad}
"""
)

research_router_prompt = PromptTemplate(
    template="""
You are a research router agent.  
You ONLY have access to the following tool:
- ArxivDownloadTool → download a research paper from arXiv into the database.

Your job:
- If the user asks about a research paper, use ArxivDownloadTool ONCE with the correct input.
- After a successful tool call, or if the paper is already available, return the Final Answer.
- Dont provide additional text after the Action Input: "" input.

---

You must follow EXACTLY one of these two options:

Option 1 (if you need to use the tool):
Question: the input question you must answer
Thought: reasoning about why to use ArxivDownloadTool
Action: ArxivDownloadTool
Action Input: <title or query string for the paper>

Rules for Option 1:
- Do NOT call the tool more than once.
- If the paper already exists in the database, do not call the tool again → go to Option 2.

---

Option 2 (if you are ready to give the final response):
Thought: I now know the final answer
Final Answer: <your detailed response here> say already existed if you see this in observation

---

Rules:
- NEVER output both an Action and a Final Answer in the same step.
- Keep the Final Answer well-explained (not minimal).

Begin!

Question: {input}
{agent_scratchpad}
"""
)
web_search_prompt = PromptTemplate(
    template="""
You are a web search based agent that must answer user queries using your knowledge and the tools provided.

You have access to the following tools:
{tools}

You must follow EXACTLY one of these two options in your output:

---

Option 1 (if you need to use a tool):
Question: the input question you must answer
Thought: reasoning about which tool to use

When you decide to use a tool, you must output ONLY the following lines (nothing else):

Action: <tool name, exactly as given in [{tool_names}]>
Action Input: <input for the tool>

Do NOT add any extra text, explanation, or commentary after Action or Action Input.

(then wait for the Observation. You may repeat this cycle up to 3 times.)

---

Option 2 (if you are ready to give the final response):
Thought: I now know the final answer
Final Answer: <your detailed and comprehensive response>

---

Rules:
- NEVER output both an Action and a Final Answer in the same step.
- Keep the Final Answer detailed and comprehensive.

Begin!

Question: {input}
{agent_scratchpad}
"""
)


In [15]:
from langchain.agents import create_react_agent, AgentExecutor, Tool
from langchain_community.utilities import DuckDuckGoSearchAPIWrapper

ddg = DuckDuckGoSearchAPIWrapper()

### --- Define Agent 1: VectorDB Expert ---
vdb_tools = [
    Tool(
        name="QuestionAnsweringFromVDBTool",
        func=question_answring_from_vdb,
        description="Answer questions from the data stored in the current vector database."
    ),
    Tool(
        name="InsightGeneratorTool",
        func=get_insights,
        description="Generate deep insights about a query based on similarity search over the vector database."
    ),
    Tool(
        name="CitationsTool",
        func=get_citations,
        description="Only retrieves citations of all papers available in the vector database."
    ),
    Tool(
        name="SummarizerTool",
        func=summarize_from_keyword,
        description="Generate summaries from retrieved content using a keyword query."
    )
]

# Fill in static values (tools + tool_names) before passing to create_react_agent
vdb_prompt = vdb_router_prompt.partial(
    tools="\n".join([f"{t.name}: {t.description}" for t in vdb_tools]),
    tool_names=", ".join([t.name for t in vdb_tools])
)

vdb_agent = create_react_agent(llama, vdb_tools, vdb_prompt)
vdb_executor = AgentExecutor(agent=vdb_agent, tools=vdb_tools, verbose=True, handle_parsing_errors=True, return_intermediate_steps=True)

### --- Define Agent 2: Research Agent ---
research_tools = [
    Tool(
        name="ArxivDownloadTool",
        func=arxiv,
        description="Search and download papers from arXiv using Keyword eg. 'Keyword', and embed them."
    )
]

research_prompt = research_router_prompt.partial(
    tools="\n".join([f"{t.name}: {t.description}" for t in research_tools]),
    tool_names=", ".join([t.name for t in research_tools])
)

research_agent = create_react_agent(llama, research_tools, research_prompt)
research_executor = AgentExecutor(agent=research_agent, tools=research_tools, verbose=True, handle_parsing_errors=True)

web_search_tools = [
    Tool(
        name="WebSearchTool",
        func=ddg.run,  # duckduckgo search wrapper
        description="Search the web for real-time information when the data is not available in the database."
    )
]

web_search_agent = create_react_agent(llm=llama, tools=web_search_tools, prompt=web_search_prompt)

web_search_executor = AgentExecutor(
    agent=web_search_agent, 
    tools=web_search_tools, 
    verbose=True, 
    handle_parsing_errors=True,
)


In [16]:
from typing import TypedDict, Any
from langgraph.graph import StateGraph, END, START

# -------------------------
# 1. Define State
# -------------------------
class MultiAgentState(TypedDict):
    question: str
    vdb_result: Any
    research_result: Any
    web_search_result: Any


# -------------------------
# 2. Define Nodes
# -------------------------
def vdb_node(state: MultiAgentState) -> MultiAgentState:
    result = vdb_executor.invoke({"input": state["question"]})
    state["vdb_result"] = result
    return state


def research_node(state: MultiAgentState) -> MultiAgentState:
    result = research_executor.invoke({"input": state["question"]})
    state["research_result"] = result
    return state


def web_search_node(state: MultiAgentState) -> MultiAgentState:
    result = web_search_executor.invoke({"input": state["question"]})
    state["web_search_result"] = result
    return state


# -------------------------
# 3. Define Router Logic
# -------------------------
def research_router(state: MultiAgentState) -> str:
    """Choose next node based on research result"""

    global vdb_available
    result = state.get("research_result")
    if not result:
        return "WebSearchAgent"

    text = str(result).lower()

    #Success > go to VDB
    if (vdb_available):
        return "VDBAgent"

    #Failure > go to WebSearch
    if "no results" in text or "unable to find" in text or "error" in text:
        return "WebSearchAgent"

    # Default fallback
    return "WebSearchAgent"


# -------------------------
# 4. Build Graph
# -------------------------
graph = StateGraph(MultiAgentState)

graph.add_node("ResearchAgent", research_node)
graph.add_node("VDBAgent", vdb_node)
graph.add_node("WebSearchAgent", web_search_node)

# Entry point
graph.add_edge(START, "ResearchAgent")

# Conditional routing (either VDB or WebSearch, not both)
graph.add_conditional_edges("ResearchAgent", research_router, {
    "VDBAgent": "VDBAgent",
    "WebSearchAgent": "WebSearchAgent"
})

# End after either branch
graph.add_edge("VDBAgent", END)
graph.add_edge("WebSearchAgent", END)

app = graph.compile()

In [19]:
user_query = input("Enter What you want to search: ")
state = {
    "question": f"{user_query}",
    "research_result": None,
    "vdb_result": None,
    "web_search_result": None,
}

final_state = app.invoke(state)

if final_state.get("vdb_result"):
    print("Final State (from VDB):\n", final_state["vdb_result"])
elif final_state.get("web_search_result"):
    print("Final State (from Web Search):\n", final_state["web_search_result"])
# elif final_state.get("research_result"):
#     print("Final State (from Research):\n", final_state["research_result"])
else:
    print("No result found")

Enter What you want to search:  explain transformers architecture




> Entering new AgentExecutor chain...
Thought: The question is about explaining the Transformers architecture. Since I don't have any information about this paper already existing in my database, I will use ArxivDownloadTool to download it.

Action: ArxivDownloadTool
Action Input: "Transformers"Got Paper Entitled:  Compress, Gather, and Recompute: REFORMing Long-Context Processing in Transformers

Stored 156 chunks in collection '2506.01215v2'
Got Paper Entitled:  Mamba Drafters for Speculative Decoding

Stored 118 chunks in collection '2506.01206v1'
Got Paper Entitled:  Humanoid World Models: Open World Foundation Models for Humanoid Robotics

Stored 114 chunks in collection '2506.01182v2'

Local collections updated with:
['2506.01215v2', '2506.01206v1', '2506.01182v2']

Available collections:
[Collection(name=2506.01206v1), Collection(name=1206.6842v1), Collection(name=2506.01215v2), Collection(name=2506.01182v2)]

Action: ArxivDownloadTool
Action Input: "Transformers"Got Paper Ent